# Прогноз временных рядов на 90 дней


Для прогноза сравним две модели:

1. `seasonal_naive_7` - повторяет значения за последнюю неделю;
2. `Holt-Winters` - учитывает тренд и недельную сезонность.

Для каждого ряда выберем модель с меньшей ошибкой `MAE` на последних 90 известных днях.

## 1. Импорт библиотек

In [10]:
from pathlib import Path
import warnings

import pandas as pd
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

## 2. Настройки

In [11]:
DATA_PATH = Path('public_data.csv')
OUTPUT_PATH = Path('forecast_90_days.csv')

FORECAST_START = '2020-05-22'
FORECAST_END = '2020-08-19'
HORIZON = 90

columns = [
    'Page_loads',
    'Unique_visits',
    'First_time_visits',
    'Returning_visits',
]

## 3. Загрузка данных

In [12]:
data = pd.read_csv(DATA_PATH, parse_dates=['Date'])
data = data.set_index('Date').asfreq('D')

print('Размер таблицы:', data.shape)
print('Период данных:', data.index.min().date(), '-', data.index.max().date())

data.head()

Размер таблицы: (2077, 4)
Период данных: 2014-09-14 - 2020-05-21


,Page_loads,Unique_visits,First_time_visits,Returning_visits
Date,,,,
2014-09-14,2146,1582,1430,152
2014-09-15,3621,2528,2297,231
2014-09-16,3698,2630,2352,278
2014-09-17,3667,2614,2327,287
2014-09-18,3316,2366,2130,236


## 4. Проверка данных

In [13]:
print('Пропуски по колонкам:')
display(data[columns].isna().sum())

expected_days = (data.index.max() - data.index.min()).days + 1
print('Даты идут без разрывов:', expected_days == len(data))

Пропуски по колонкам:


Page_loads           0
Unique_visits        0
First_time_visits    0
Returning_visits     0
dtype: int64

Даты идут без разрывов: True


## 5. Метрики и две базовые модели

Для сравнения моделей используем `MAE` - среднюю абсолютную ошибку. Чем меньше значение, тем лучше прогноз.

In [14]:
def seasonal_naive_7(train, steps):
    last_week = train.iloc[-7:].to_numpy()
    forecast = [last_week[i % 7] for i in range(steps)]
    return pd.Series(forecast).clip(lower=0).round().astype(int)


def holt_winters(train, steps):
    model = ExponentialSmoothing(
        train,
        trend='add',
        damped_trend=True,
        seasonal='add',
        seasonal_periods=7,
        initialization_method='estimated',
    )

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fitted_model = model.fit(optimized=True, remove_bias=True)

    return fitted_model.forecast(steps).clip(lower=0).round().astype(int)


models = {
    'seasonal_naive_7': seasonal_naive_7,
    'holt_winters': holt_winters,
}

## 6. Сравнение моделей

Берем последние 90 дней из имеющихся данных как проверочный период. Модели обучаются на более ранних данных, затем прогнозируют эти 90 дней.

In [15]:
train = data.iloc[:-HORIZON]
test = data.iloc[-HORIZON:]

scores = []
best_models = {}

for column in columns:
    column_scores = []

    for model_name, model_func in models.items():
        pred = model_func(train[column], HORIZON)
        error = mean_absolute_error(test[column], pred)

        column_scores.append((model_name, error))
        scores.append({
            'ряд': column,
            'модель': model_name,
            'MAE': round(error, 2),
        })

    best_models[column] = min(column_scores, key=lambda x: x[1])[0]

scores = pd.DataFrame(scores)

display(scores)
display(pd.Series(best_models, name='лучшая модель').to_frame())

,ряд,модель,MAE
0,Page_loads,seasonal_naive_7,696.29
1,Page_loads,holt_winters,776.78
2,Unique_visits,seasonal_naive_7,519.78
3,Unique_visits,holt_winters,560.54
4,First_time_visits,seasonal_naive_7,433.07
5,First_time_visits,holt_winters,475.97
6,Returning_visits,seasonal_naive_7,96.69
7,Returning_visits,holt_winters,103.84


,лучшая модель
Page_loads,seasonal_naive_7
Unique_visits,seasonal_naive_7
First_time_visits,seasonal_naive_7
Returning_visits,seasonal_naive_7


## 7. Финальный прогноз

Теперь обучаем выбранные модели на всех данных и прогнозируем нужные 90 дней.

In [16]:
forecast_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')
steps = len(forecast_dates)

forecast = pd.DataFrame(index=forecast_dates)

for column in columns:
    model_name = best_models[column]
    forecast[column] = models[model_name](data[column], steps).to_numpy()

print('Количество строк прогноза:', len(forecast))
forecast.head()

Количество строк прогноза: 90


,Page_loads,Unique_visits,First_time_visits,Returning_visits
2020-05-22,4958,3829,3183,646
2020-05-23,3651,2827,2384,443
2020-05-24,4248,3291,2737,554
2020-05-25,5624,4231,3588,643
2020-05-26,5595,4300,3666,634


## 8. Просмотр результата

In [17]:
forecast.tail()

,Page_loads,Unique_visits,First_time_visits,Returning_visits
2020-08-15,3651,2827,2384,443
2020-08-16,4248,3291,2737,554
2020-08-17,5624,4231,3588,643
2020-08-18,5595,4300,3666,634
2020-08-19,5213,4106,3466,640


## 9. Сохранение результата

Сохраняем прогноз в CSV-файл.

In [18]:
result = forecast.copy()
result.insert(0, 'Date', result.index.date.astype(str))

result.to_csv(OUTPUT_PATH, index=False)

print('Файл сохранен:', OUTPUT_PATH)
print('Период прогноза:', result['Date'].iloc[0], '-', result['Date'].iloc[-1])
print('Количество строк:', len(result))

result

Файл сохранен: forecast_90_days.csv
Период прогноза: 2020-05-22 - 2020-08-19
Количество строк: 90


,Date,Page_loads,Unique_visits,First_time_visits,Returning_visits
2020-05-22,2020-05-22,4958,3829,3183,646
2020-05-23,2020-05-23,3651,2827,2384,443
2020-05-24,2020-05-24,4248,3291,2737,554
2020-05-25,2020-05-25,5624,4231,3588,643
2020-05-26,2020-05-26,5595,4300,3666,634
...,...,...,...,...,...
2020-08-15,2020-08-15,3651,2827,2384,443
2020-08-16,2020-08-16,4248,3291,2737,554
2020-08-17,2020-08-17,5624,4231,3588,643
2020-08-18,2020-08-18,5595,4300,3666,634
